### 0. Imports libs and auxiliary functions

In [4]:
import sys, os        

# --- Core libs ---
import numpy as np
import pandas as pd
from pathlib import Path
import glob, tempfile

# --- HuggingFace (BERT) ---
from transformers import BertTokenizerFast, BertModel
import torch

# --- Progress bar ---
from tqdm import tqdm
tqdm.pandas()

# --- spaCy compatibility (older TAACO/TAALED expect this) ---
import spacy
if not hasattr(spacy.util, "set_data_path"):
    spacy.util.set_data_path = lambda *a, **k: None

# --- TAASSC (your local module) ---
import taassc as tdev
from taassc import LGR_Analysis, index_list

# --- TAALED setup (needs GUI stubs) ---
import taaled as TAALED
class _Root:
    def update_idletasks(self): pass
TAALED.root = _Root()
TAALED.system = "L" 

# --- TAACO setup (import + GUI stubs + resource path) ---
import TAACO.TAACOnoGUI as TAACOnoGUI
from TAACO.TAACOnoGUI import runTAACO
TAACOnoGUI.root = _Root()
TAACOnoGUI.system = "L"

/Users/pedropertusi/Desktop/ContentXForm/env/lib/python3.12/site-packages/lexical_diversity/lex_div.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [5]:
def read_df(path: str) -> pd.DataFrame:
    """Read DataFrame from CSV or Parquet based on file extension."""
    if path.endswith(".parquet"):
        return pd.read_parquet(path)
    return pd.read_csv(path)

def save_csv(df, path):
    """Save DataFrame as CSV and print confirmation."""
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)
    print(f"Saved: {path}")

---- 

### 1. Define witch data we are preprocessing (input data)
- Original data source (includes cleaning + preprocessing steps)
- Rewritten text/data (only preprocessing steps)

##### Setting up the reading and saving directory
- Input data choice: original/rewritten
- Input directory: where the data is stored (original/rewritten)
- Output directory: where the preprocessed data will be saved

In [6]:
# ==== RUN SETTINGS ====
TEXT_COL = "text"

# Original data
DF_PATH = "../data/raw/persuade_2.csv"
SAVE_LOW_HIGH = True
COLUMNS_KEEP = ['full_text', 'holistic_essay_score', 'race_ethnicity', 'gender', 'grade_level', 'economically_disadvantaged', 'prompt_name']

# Outputs
EMB_DIR = "../outputs/embeddings/original/"
OUT_DIR = "../data/processed/original/"

print("Creating directory structure for output...")
for folder in [EMB_DIR, OUT_DIR]:
    os.makedirs(folder, exist_ok=True)
    print(f"Verified: {folder}")
print("\nAll directories are ready.")

RUN_TAALED = True
RUN_TAACO = True
RUN_TAASSC = True

Creating directory structure for output...
Verified: ../outputs/embeddings/original/
Verified: ../data/processed/original/

All directories are ready.


---- 

### 2. Loading Dataset

In [7]:
print("Loading original dataset")
df = read_df(DF_PATH)
df.head()

Loading original dataset


,essay_id_comp,full_text,holistic_essay_score,word_count,prompt_name,task,assignment,source_text,gender,grade_level,ell_status,race_ethnicity,economically_disadvantaged,student_disability_status
0,423A1CA112E2,Phones\n\nModern humans today are always on th...,3,378,Phones and driving,Independent,Today the majority of humans own and operate c...,NaN,M,NaN,NaN,Black/African American,NaN,NaN
1,BC75783F96E3,This essay will explain if drivers should or s...,4,432,Phones and driving,Independent,Today the majority of humans own and operate c...,NaN,M,NaN,NaN,Black/African American,NaN,NaN
2,74C8BC7417DE,Driving while the use of cellular devices\n\nT...,2,179,Phones and driving,Independent,Today the majority of humans own and operate c...,NaN,F,NaN,NaN,White,NaN,NaN
3,A8445CABFECE,Phones & Driving\n\nDrivers should not be able...,3,221,Phones and driving,Independent,Today the majority of humans own and operate c...,NaN,M,NaN,NaN,Black/African American,NaN,NaN
4,6B4F7A0165B9,Cell Phone Operation While Driving\n\nThe abil...,4,334,Phones and driving,Independent,Today the majority of humans own and operate c...,NaN,M,NaN,NaN,White,NaN,NaN


-----

### 3. Data cleaning and feature selection

In [8]:
df = df[COLUMNS_KEEP]
df = df.dropna()
df.reset_index(drop=True, inplace=True)
df = df.rename(columns={'full_text': 'text'})
df['economically_disadvantaged'] = df['economically_disadvantaged'].map({'Economically disadvantaged': 1, 'Not economically disadvantaged': 0})

-----

### 4. Data Preprocessing Steps (NLP tools)
- TAALED
- TAACCO
- TAASSCC

### TAALED

In [9]:
if not hasattr(spacy.util, "set_data_path"):
    spacy.util.set_data_path = lambda *a, **k: None
class _Root:
    def update_idletasks(self):  
        pass

TAALED.root = _Root()  # satisfy TAALED's references to a Tk root
TAALED.system = "L"    # pretend we're on Linux/Mac ('L' or 'M'); avoids GUI branches

In [ ]:
# ==== TAALED runner (always 'taaled_' prefix) ====
TAALED_VAR_DICT = {
    "aw": 1, "cw": 1, "fw": 1,
    "simple_ttr": 1, "root_ttr": 1, "log_ttr": 1, "maas_ttr": 1,
    "mattr": 1, "msttr": 1, "hdd": 1,
    "mltd": 1, "mltd_ma": 1, "mtld_wrap": 1,
    "indout": 0,
}

def _detect_filename_col(res):
    # Find filename column in TAALED's CSV
    cands = {"filename","file","file_name","textname","doc","document","name"}
    for c in res.columns:
        if c.lower() in cands:
            return c
    return res.columns[0]

def _run_taaled_once(df, *, text_col, var_dict):
    # Run TAALED on one DataFrame and merge taaled_* metrics back
    if text_col not in df.columns:
        raise ValueError(f"TEXT_COL='{text_col}' not found in columns: {list(df.columns)}")
    df = df.copy()

    with tempfile.TemporaryDirectory() as tmp_dir:
        for i, txt in df[text_col].items():
            with open(os.path.join(tmp_dir, f"{i}.txt"), "w", encoding="utf-8") as f:
                f.write(txt if isinstance(txt, str) else "")

        out_csv = os.path.join(tmp_dir, "taaled_out.csv")
        import tkinter.messagebox as mb

        def _no_popup(*args, **kwargs):
            # do nothing instead of showing a dialog
            return None

        mb.showinfo = _no_popup
        
        TAALED.main(tmp_dir, out_csv, var_dict)
        res = pd.read_csv(out_csv)

    fn_col = _detect_filename_col(res)
    res["__idx__"] = res[fn_col].astype(str).str.replace(".txt", "", regex=False)
    df["__idx__"] = df.index.astype(str)

    metric_cols = [c for c in res.columns if c not in (fn_col, "__idx__")]
    res = res.rename(columns={c: f"taaled_{c}" for c in metric_cols})

    merged = df.merge(res.drop(columns=[fn_col]), on="__idx__", how="left")
    return merged.drop(columns="__idx__")

def run_taaled(df_or_list, *, mode, text_col, var_dict):
    if mode == "original":
        return _run_taaled_once(df_or_list, text_col=text_col, var_dict=var_dict)
    elif mode == "rewrites":
        return [_run_taaled_once(d, text_col=text_col, var_dict=var_dict) for d in df_or_list]
    else:
        raise ValueError("mode must be 'original' or 'rewrites'")

# ---- EXECUTE ----
if RUN_TAALED:
    df = run_taaled(df, mode="original", text_col=TEXT_COL, var_dict=TAALED_VAR_DICT)
    print("TAALED done.")
else:
    print("RUN_TAALED is False — skipping.")

------

### TAACO

In [ ]:
opts = {
    "sourceKeyOverlap": False, "sourceLSA": False, "sourceLDA": False, "sourceWord2vec": False,
    "wordsAll": True, "wordsContent": True, "wordsFunction": True,
    "wordsNoun": True, "wordsPronoun": True, "wordsArgument": True,
    "wordsVerb": True, "wordsAdjective": True, "wordsAdverb": True,
    "overlapSentence": True, "overlapParagraph": True,
    "overlapAdjacent": True, "overlapAdjacent2": True,
    "otherTTR": True, "otherConnectives": True, "otherGivenness": True,
    "overlapLSA": True, "overlapLDA": True, "overlapWord2vec": True,
    "overlapSynonym": True, "overlapNgrams": True,
    "outputTagged": False, "outputDiagnostic": False,
}

# ==== TAACO runner (simplified, always 'taaco_' prefix) ====

def _detect_taaco_filename_col(res):
    """Find which column in TAACO's CSV contains the filenames."""
    candidates = {"filename","file","file_name","textname","doc","document","name"}
    for c in res.columns:
        if c.lower() in candidates:
            return c
    for c in res.columns:  # fallback: looks like '*.txt'
        try:
            if res[c].astype(str).str.endswith(".txt").any():
                return c
        except Exception:
            pass
    return res.columns[0]

def _run_taaco_once(df, *, text_col, opts):
    """Run TAACO on a single DataFrame and merge taaco_ metrics back."""
    if text_col not in df.columns:
        raise ValueError(f"TEXT_COL='{text_col}' not found")
    df = df.copy()

    with tempfile.TemporaryDirectory() as tmp_dir:
        for i, txt in df[text_col].items():
            with open(os.path.join(tmp_dir, f"{i}.txt"), "w", encoding="utf-8") as f:
                f.write(txt if isinstance(txt, str) else "")

        out_csv = os.path.join(tmp_dir, "taaco_out.csv")
        runTAACO(tmp_dir, out_csv, opts)
        res = pd.read_csv(out_csv)

    fn_col = _detect_taaco_filename_col(res)
    res["__idx__"] = res[fn_col].astype(str).str.replace(".txt", "", regex=False)
    df["__idx__"] = df.index.astype(str)

    metric_cols = [c for c in res.columns if c not in (fn_col, "__idx__")]
    res = res.rename(columns={c: f"taaco_{c}" for c in metric_cols})

    merged = df.merge(res.drop(columns=[fn_col]), on="__idx__", how="left")
    return merged.drop(columns="__idx__")

def run_taaco(df_or_list, *, mode, text_col, opts):
    """Handles both modes: DataFrame (original) or list[DataFrame] (rewrites)."""
    if mode == "original":
        return _run_taaco_once(df_or_list, text_col=text_col, opts=opts)
    elif mode == "rewrites":
        return [_run_taaco_once(d, text_col=text_col, opts=opts) for d in df_or_list]
    else:
        raise ValueError("mode must be 'original' or 'rewrites'")

# ---- EXECUTE ----
if RUN_TAACO:
    # Save current working directory and switch into TAACO folder
    orig_cwd = os.getcwd()
    taaco_dir = "TAACO"  
    os.chdir(taaco_dir)

    try:
        df = run_taaco(df, mode="original", text_col=TEXT_COL, opts=opts)
        print("TAACO done.")

    finally:
        # Always restore the original directory even if TAACO crashes
        os.chdir(orig_cwd)
        print(f"→ reverted working dir to: {os.getcwd()}")

----

### TAASSC

In [ ]:
# ==== TAASSC runner (simplified, always 'taassc_' prefix) ====

def _run_taassc_once(df, *, text_col, index_list):
    """Run TAASSC (LGR_Analysis) on one DataFrame and merge 'taassc_' metrics back."""
    if text_col not in df.columns:
        raise ValueError(f"TEXT_COL='{text_col}' not found in columns: {list(df.columns)[:12]}...")
    df = df.copy()

    records = []
    for txt in df[text_col].fillna(""):
        try:
            res = LGR_Analysis(txt)  # dict of TAASSC metrics
            row = {m: res.get(m, float('nan')) for m in index_list}
        except Exception:
            row = {m: float('nan') for m in index_list}
        records.append(row)

    metrics_df = pd.DataFrame.from_records(records, index=df.index)
    metrics_df.columns = [f"taassc_{m}" for m in index_list]

    return pd.concat([df, metrics_df], axis=1)

def run_taassc(df_or_list, *, mode, text_col, index_list):
    """Handles both modes: DataFrame (original) or list[DataFrame] (rewrites)."""
    if mode == "original":
        return _run_taassc_once(df_or_list, text_col=text_col, index_list=index_list)
    elif mode == "rewrites":
        return [_run_taassc_once(d, text_col=text_col, index_list=index_list) for d in df_or_list]
    else:
        raise ValueError("mode must be 'original' or 'rewrites'")

# ---- EXECUTE ----
if RUN_TAASSC:
    df = run_taassc(df, mode="original", text_col=TEXT_COL, index_list=index_list)
    print("TAASSC done.")
else:
    print("RUN_TAASSC is False — skipping.")

-----

### 4. Save Preprocessed Data

In [ ]:
# ==== SAVE CLEANED DATASETS (CSV only) ====
print("Saving original dataset and SES splits...")

# Full version
save_csv(df, OUT_DIR + "original_full.csv")

# Low / High SES splits
if SAVE_LOW_HIGH:
    low = df[df["economically_disadvantaged"] == 1].reset_index(drop=True)
    high = df[df["economically_disadvantaged"] == 0].reset_index(drop=True)
    save_csv(low, OUT_DIR + "original_low_SES.csv")
    save_csv(high, OUT_DIR + "original_high_SES.csv")

print("All saves complete.")


----

### Embeddings

In [ ]:
device = torch.device('mps') if (torch.backends.mps.is_available()) else torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = BertTokenizerFast.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased').eval().to(device)

def get_embeddings(text: str) -> np.ndarray:
    """
    Run one text through BERT, return the [CLS] embedding as a numpy vector.
    """
    inputs = tokenizer(text,
                       return_tensors='pt',
                       padding=True,
                       truncation=True,
                       max_length=512)
    # move to device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
    # [batch=1, seq, dim] → pick CLS token embedding
    cls_emb = outputs.last_hidden_state[0, 0, :]
    return cls_emb.cpu().numpy()

# ==== EMBEDDINGS SAVE (BERT [CLS]) ====
def _embed_df(df, text_col):
    texts = df[text_col].fillna("").tolist()
    it = texts
    if "tqdm" in globals():  # optional progress bar if you already imported tqdm
        it = tqdm(texts, desc="Embedding", total=len(texts))
    embs = [get_embeddings(t) for t in it]
    return np.vstack(embs) if len(embs) else np.zeros((0, model.config.hidden_size), dtype=float)

In [ ]:
EMB_DIR = Path(EMB_DIR)

X_full = _embed_df(df, TEXT_COL)
np.save(EMB_DIR / "embeddings_original_full.npy", X_full)

if SAVE_LOW_HIGH:
    low = df[df["economically_disadvantaged"] == 1].reset_index(drop=True)
    high = df[df["economically_disadvantaged"] == 0].reset_index(drop=True)

    X_low = _embed_df(low, TEXT_COL) if len(low) else np.zeros((0, X_full.shape[1]), dtype=float)
    X_high = _embed_df(high, TEXT_COL) if len(high) else np.zeros((0, X_full.shape[1]), dtype=float)

    np.save(EMB_DIR / "embeddings_original_low.npy", X_low)
    np.save(EMB_DIR / "embeddings_original_high.npy", X_high)

    print(f"Saved embeddings to {EMB_DIR}")